In [1]:
import os
import sys

# Força o Python a usar os certificados que acabamos de instalar no Conda
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()

# Desativa o motor Xet e o suporte a links simbólicos (comum dar erro em Singularity)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import pandas as pd
import ast
from datasets import load_dataset

print("Iniciando Fase 1: Carregamento e Tratamento Lexical (Revisão Gramatical e Afetiva)")

# --- Parâmetros de Qualidade ---
LIMIAR_CONSISTENCIA_EQ = 0.50  
TOP_K_PALAVRAS = 100           

dataset_eq = load_dataset("seungheondoh/socialfx-original", split="eq").to_pandas()
dataset_reverb = load_dataset("seungheondoh/socialfx-original", split="reverb").to_pandas()

def parse_extra_data(extra_str):
    if isinstance(extra_str, str):
        try: return ast.literal_eval(extra_str)
        except: return {}
    return extra_str if isinstance(extra_str, dict) else {}

def parse_params(param_val):
    if isinstance(param_val, str):
        try: return ast.literal_eval(param_val)
        except: return []
    return param_val

def process_socialfx_split(df, split_name, top_k):
    df['extra_dict'] = df['extra'].apply(parse_extra_data)
    df['lang'] = df['extra_dict'].apply(lambda x: x.get('lang', ''))
    
    if split_name == 'eq':
        df['consistency'] = df['extra_dict'].apply(lambda x: x.get('ratings_consistency', 0))
        df_filtered = df[(df['lang'] == 'English') & (df['consistency'] >= LIMIAR_CONSISTENCIA_EQ)].copy()
        
        # Filtro de Contexto (Blacklist focada em anomalias literais; 'love', 'happy', 'good' mantidos)
        blacklist = ['taco', 'baseball', 'disgusting', 'genius', 'poor']
        df_filtered = df_filtered[~df_filtered['text'].isin(blacklist)]
        
        df_filtered = df_filtered.sort_values(by='consistency', ascending=False)
        
    elif split_name == 'reverb':
        def calc_consensus_strength(row_dict):
            agreed = row_dict.get('agreed', '')
            if not agreed: return 0
            agreed_list = [w.strip() for w in agreed.split(',') if w.strip() not in ['', 'none', 'none of the above']]
            return len(agreed_list)

        df['consensus_strength'] = df['extra_dict'].apply(calc_consensus_strength)
        df_filtered = df[(df['lang'] == 'English') & (df['consensus_strength'] > 0)].copy()
        
        # Filtro Gramatical: Permite no máximo 2 vírgulas (cadeia de até 3 palavras)
        df_filtered = df_filtered[df_filtered['text'].str.count(',') <= 2]
        
        # Otimização NLP: Adiciona espaço após a vírgula para formatação natural no prompt
        df_filtered['text'] = df_filtered['text'].str.replace(',', ', ')
        
        df_filtered = df_filtered.sort_values(by='consensus_strength', ascending=False)

    df_unique = df_filtered.drop_duplicates(subset=['text'], keep='first').copy()
    df_unique['param_values_list'] = df_unique['param_values'].apply(parse_params)
    
    if len(df_unique) > top_k:
        df_unique = df_unique.head(top_k)
        
    return df_unique

df_eq_ouro = process_socialfx_split(dataset_eq, 'eq', top_k=TOP_K_PALAVRAS)
df_reverb_ouro = process_socialfx_split(dataset_reverb, 'reverb', top_k=TOP_K_PALAVRAS)

# --- Saída de Dados e Amostragem de Validação ---
print("\nProcessamento concluído.")
print(f"-> EQ (Top-{TOP_K_PALAVRAS})   : {len(df_eq_ouro)} palavras únicas.")
print(f"-> Reverb (Top-{TOP_K_PALAVRAS}) : {len(df_reverb_ouro)} palavras únicas.")

print("\n--- AMOSTRAGEM DE VALIDAÇÃO (Limites do Top-100) ---")

print("\nEQUALIZAÇÃO (Métrica: Consistência Numérica)")
print(f"  Maior Consistência (Top 1) : '{df_eq_ouro.iloc[0]['text']}' | Score: {df_eq_ouro.iloc[0]['consistency']:.4f}")
print(f"  Menor Consistência (Top {len(df_eq_ouro)}): '{df_eq_ouro.iloc[-1]['text']}' | Score: {df_eq_ouro.iloc[-1]['consistency']:.4f}")

print("\nREVERBERAÇÃO (Métrica: Densidade Semântica)")
print(f"  Maior Densidade (Top 1)    : '{df_reverb_ouro.iloc[0]['text']}' | Score: {df_reverb_ouro.iloc[0]['consensus_strength']}")
print(f"  Menor Densidade (Top {len(df_reverb_ouro)}): '{df_reverb_ouro.iloc[-1]['text']}' | Score: {df_reverb_ouro.iloc[-1]['consensus_strength']}")

Iniciando Fase 1: Carregamento e Tratamento Lexical (Revisão Gramatical e Afetiva)



Processamento concluído.
-> EQ (Top-100)   : 100 palavras únicas.
-> Reverb (Top-100) : 100 palavras únicas.

--- AMOSTRAGEM DE VALIDAÇÃO (Limites do Top-100) ---

EQUALIZAÇÃO (Métrica: Consistência Numérica)
  Maior Consistência (Top 1) : 'deep' | Score: 0.9314
  Menor Consistência (Top 100): 'whispered' | Score: 0.7180

REVERBERAÇÃO (Métrica: Densidade Semântica)
  Maior Densidade (Top 1)    : 'beautiful-mixing' | Score: 11
  Menor Densidade (Top 100): 'dry, hard, drum' | Score: 6


In [2]:
# --- Bloco Opcional: Inspeção Completa do Top-100 ---
print("\n" + "="*85)
print("🏆 LISTA COMPLETA: TOP-100 VOCABULÁRIO DE OURO 🏆")
print("="*85)

# Cabeçalho da tabela
print(f"{'EQUALIZAÇÃO (Consistência)'.ljust(40)} | {'REVERBERAÇÃO (Densidade)'}")
print("-" * 85)

# Itera sobre os 100 índices e imprime lado a lado
for i in range(TOP_K_PALAVRAS):
    # Prepara a string de EQ
    if i < len(df_eq_ouro):
        eq_word = df_eq_ouro.iloc[i]['text']
        eq_score = df_eq_ouro.iloc[i]['consistency']
        eq_str = f"#{i+1:03d} {eq_word} ({eq_score:.2f})"
    else:
        eq_str = ""

    # Prepara a string de Reverb
    if i < len(df_reverb_ouro):
        rev_word = df_reverb_ouro.iloc[i]['text']
        rev_score = df_reverb_ouro.iloc[i]['consensus_strength']
        # Reverb usa score inteiro (contagem de palavras), não float
        rev_str = f"#{i+1:03d} {rev_word} ({int(rev_score)})" 
    else:
        rev_str = ""

    # Imprime a linha formatada
    print(f"{eq_str.ljust(40)} | {rev_str}")


🏆 LISTA COMPLETA: TOP-100 VOCABULÁRIO DE OURO 🏆
EQUALIZAÇÃO (Consistência)               | REVERBERAÇÃO (Densidade)
-------------------------------------------------------------------------------------
#001 deep (0.93)                         | #001 beautiful-mixing (11)
#002 soft (0.93)                         | #002 epic, heavenly (10)
#003 rich (0.92)                         | #003 beautiful (9)
#004 passionate (0.91)                   | #004 statics, loud (9)
#005 metallic (0.91)                     | #005 spacious, deep, loud (9)
#006 energetic (0.90)                    | #006 haunting, frightening (9)
#007 warm (0.90)                         | #007 sweetest (9)
#008 loud (0.90)                         | #008 pleasant (8)
#009 tinny (0.89)                        | #009 spacious, distant, reflective (8)
#010 slow (0.89)                         | #010 echo (8)
#011 biting (0.88)                       | #011 close, warmth (8)
#012 low (0.88)                          | #012 sad, drea

In [3]:
import numpy as np
from pedalboard import Pedalboard, PeakFilter, Reverb

print("Iniciando Fase 2: Construção do Motor DSP Unificado")

# Configuração Base
freqs_eq = np.logspace(np.log10(20), np.log10(20000), num=40)
LIMITE_CLIPPING = 0.95

def processar_dsp_monotonico(audio_data, sample_rate, tipo_efeito, parametros, intensidade):
    """
    Motor unificado que roteia o processamento baseado no tipo de efeito (EQ ou Reverb)
    e garante a normalização de pico para evitar distorção digital.
    """
    
    # Roteamento de Equalização
    if tipo_efeito == 'eq':
        filtros = []
        for i, freq in enumerate(freqs_eq):
            # A curva base (parametros) é multiplicada pela intensidade
            ganho_escalado = parametros[i] * intensidade
            if abs(ganho_escalado) > 0.1: 
                filtros.append(PeakFilter(cutoff_frequency_hz=freq, gain_db=ganho_escalado, q=1.5))
        board = Pedalboard(filtros)
        
    # Roteamento de Reverberação
    elif tipo_efeito == 'reverb':
        # Para Reverb, os 'parametros' do SocialFX não são bandas numéricas.
        # A intensidade controla diretamente a geometria da sala e a proporção do efeito.
        room_size_calc = intensidade * 0.8  # Limita o tamanho máximo da sala a 80% para evitar feedback
        wet_level_calc = intensidade * 0.7  # Limita a mistura do efeito a 70%
        board = Pedalboard([
            Reverb(room_size=room_size_calc, wet_level=wet_level_calc, dry_level=1.0)
        ])
        
    else:
        raise ValueError("Tipo de efeito desconhecido. Use 'eq' ou 'reverb'.")

    # Conversão para mono, caso o áudio original seja estéreo
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    # Processamento da cadeia de sinal
    audio_processado = board(audio_data, sample_rate)

    # Normalização de Pico (Anti-Clipping)
    pico_maximo = np.max(np.abs(audio_processado))
    if pico_maximo > LIMITE_CLIPPING:
        audio_processado = (audio_processado / pico_maximo) * LIMITE_CLIPPING

    return audio_processado

print("Motor DSP Unificado compilado em memória com proteção Anti-Clipping.")

Iniciando Fase 2: Construção do Motor DSP Unificado
Motor DSP Unificado compilado em memória com proteção Anti-Clipping.


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm import tqdm

print("Iniciando Fase 3: Geração Matricial (Cross-Word Augmentation e Semântica)")

# --- Configurações de Escala ---
PALAVRAS_EQ_POR_NOTA = 3      
PALAVRAS_REVERB_POR_NOTA = 3  
NOTAS_LIMITE_TESTE = None        
INTENSIDADES = [0.2, 0.4, 0.6, 0.8, 1.0]

# --- Diretórios ---
dataset_dir = 'dataset_sintetico_v2_final'
os.makedirs(dataset_dir, exist_ok=True)
nsynth_audio_dir = 'nsynth-test/audio'
nsynth_json_path = 'nsynth-test/examples.json'

with open(nsynth_json_path, 'r') as f:
    nsynth_meta = json.load(f)

notas_disponiveis = list(nsynth_meta.keys())
if NOTAS_LIMITE_TESTE:
    notas_disponiveis = notas_disponiveis[:NOTAS_LIMITE_TESTE]

dataset_metadata = []

# --- Dicionário de Semântica Contínua ---
MODIFICADORES = {
    0.2: ["slightly", "faintly", "a hint of"],
    0.4: ["mildly", "a bit", "marginally"],
    0.6: ["moderately", "somewhat", "partially"],
    0.8: ["mostly", "substantially", "largely"],
    1.0: ["", "", ""] 
}

def gerar_prompt_semantico(palavra, contexto, intensidade, tipo_efeito):
    modificador = random.choice(MODIFICADORES[intensidade])
    prefixo = f"{modificador} {palavra}".strip()
    
    if tipo_efeito == 'eq':
        templates = [
            f"a {prefixo} {contexto}",
            f"{contexto} with a {prefixo} equalization",
            f"a recording of a {contexto} modified to sound {prefixo}"
        ]
    elif tipo_efeito == 'reverb':
        templates = [
            f"a {contexto} with {prefixo} reverberation",
            f"a {contexto} placed in a {prefixo} acoustic space",
            f"{contexto} processed to have a {prefixo} spatial effect"
        ]
    return random.choice(templates)

# --- Laço de Processamento Matricial ---
for nota_id in tqdm(notas_disponiveis, desc="Processando Notas NSynth"):
    
    info = nsynth_meta[nota_id]
    instrumento = f"{info['instrument_source_str']} {info['instrument_family_str']}"
    qualidades = info.get('qualities_str', [])
    qualidade_base = f"{qualidades[0]} " if qualidades else ""
    contexto_acustico = f"{qualidade_base}{instrumento}"
    
    audio_path = os.path.join(nsynth_audio_dir, f"{nota_id}.wav")
    try:
        audio_data, sr = sf.read(audio_path)
    except FileNotFoundError:
        continue
        
    amostras_eq = df_eq_ouro.sample(n=PALAVRAS_EQ_POR_NOTA)
    amostras_reverb = df_reverb_ouro.sample(n=PALAVRAS_REVERB_POR_NOTA)
    
    lote_processamento = []
    for _, row in amostras_eq.iterrows():
        lote_processamento.append(('eq', row['text'], row['param_values_list']))
    for _, row in amostras_reverb.iterrows():
        lote_processamento.append(('reverb', row['text'], None)) 
        
    for tipo_efeito, palavra_alvo, parametros in lote_processamento:
        for i in INTENSIDADES:
            pct = int(i * 100)
            
            # Requer importação e declaração do motor DSP (Fase 2) previamente
            audio_processado = processar_dsp_monotonico(
                audio_data=audio_data, 
                sample_rate=sr, 
                tipo_efeito=tipo_efeito, 
                parametros=parametros, 
                intensidade=i
            )
            
            nome_arquivo = f"{tipo_efeito}_{palavra_alvo.replace(' ', '_').replace(',', '')}_{pct}pct_{nota_id}.wav"
            caminho_salvo = os.path.join(dataset_dir, nome_arquivo)
            sf.write(caminho_salvo, audio_processado, sr)
            
            prompt = gerar_prompt_semantico(palavra_alvo, contexto_acustico, i, tipo_efeito)
            
            dataset_metadata.append({
                "file_name": nome_arquivo,
                "text": prompt,
                "keyword": palavra_alvo,
                "effect_type": tipo_efeito,
                "intensity": i,
                "original_note": nota_id
            })

df_treino_final = pd.DataFrame(dataset_metadata)
df_treino_final.to_csv(os.path.join(dataset_dir, "metadata.csv"), index=False)

print("\nGeração da Matriz de Dados concluída.")

Iniciando Fase 3: Geração Matricial (Cross-Word Augmentation e Semântica)


Processando Notas NSynth:   1%|▋                                                    | 52/4096 [01:34<2:02:43,  1.82s/it]